[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/agent-observability-course/06_autoinstrumentacion/06_proyecto_autoinstrumentacion.ipynb)

# Proyecto 06 · Manual frente a auto en SupportOps: medir, blindar y decidir

| | |
|---|---|
| **Nivel** | 🟡 Intermedio |
| **Duración estimada** | ≈ 1 h 45 min |
| **Servicios de Compose** | Ninguno (conjunto pregrabado; funciona en Colab) |
| **RAM aproximada** | < 500 MB |
| **Prerrequisitos** | Lección `06_autoinstrumentacion` (laboratorio, helpers de análisis, `SpanProcessor`, matriz) y módulos 03-05 |
| **Conexión** | Convierte el experimento de la lección en **tres herramientas de ingeniería** que usarías de verdad: un inventario de ruido, un detector de doble instrumentación/fugas de PII y una **prueba de contrato** para actualizar librerías con seguridad. Prepara el módulo 07 (qué consultas podrás escribir en TraceQL) y el 20 (privacidad) |

## Contexto de negocio

Eres ingeniero/a de plataforma en **SupportOps**. El equipo de producto quiere «activar OpenLIT/OpenLLMetry y listo». Tu jefe te pide una **recomendación defendible con datos** y una red de seguridad para que ninguna actualización de librería rompa los *dashboards* ni filtre datos de clientes. Dispones de la telemetría de 7 estrategias sobre los mismos 24 tickets (semilla 42): manual, OpenLIT, OpenLLMetry, oficial de OpenTelemetry, híbrido oficial, híbrido OpenLLMetry y OpenLLMetry con decoradores, además de las pruebas de trampas (doble instrumentación, reintentos del SDK).

## Datos

- **Pregrabados** (`recorded/lab_seed42.json.gz`, generados con OpenLIT 1.45.0, traceloop-sdk 0.62.4, genai-openai 1.2b0, openai 2.54.0, OpenTelemetry SDK 1.45.1): cada ejecución = `{"spans": [...], "results": [...], "meta": {...}}`; los spans siguen `supportops.schema.SPAN_SCHEMA` (+ `scope`); `results` es el `AgentResult` de cada ticket (verdad de campo); `meta["server_requests"]` es el número de peticiones que recibió de verdad el servidor.
- Todo es **determinista**: los recuentos de spans no dependen de tu máquina ni de las latencias. (Si re-generas el conjunto con otras versiones de las librerías, las cifras pueden cambiar: esa es justo la razón de la prueba de contrato.)
- Si instalaste las librerías, `lab.run_variant(...)` puede reproducir cualquier ejecución (retos extra).

## Entregables y criterios de evaluación (rúbrica)

| # | Entregable | Criterio medido | Umbral (calibrado con la solución, dataset pregrabado, semilla 42) |
|---|---|---|---|
| R1 | `span_inventory` | Recuento por categoría (agent / chat / tool / http / other) en `openlit`, `hybrid_openllmetry` y `openllmetry_decorated` | coincidencia exacta en 3/3 |
| R2 | `duplicate_ratio` | Spans `chat` / peticiones reales: `both` = 2,0 · `official` = 1,0 · `retries2:official` ≈ 0,686 | tolerancia ±0,005 en 3/3 |
| R3 | `leak_report` | Spans con PII, correos distintos y **atributos** por los que se filtra (`openllmetry_decorated`, `off:openlit_decorated`, `official`) | coincidencia exacta en 3/3 |
| R4 | `TicketContextProcessor` | En el banco de pruebas: 48 de 49 spans con `supportops.ticket.id` (el que se creó **fuera** de un ticket no lo lleva) y 49 de 49 con `supportops.instrumented_by` | exacto |
| R5 | `contract_violations` | Violaciones del contrato en `openlit`, `openllmetry`, `official`, `hybrid_official` | coincidencia exacta en 4/4 |
| R6 | `weighted_scores` + `win_rates` + `DECISION` | Ganador por perfil, tasa de victorias del híbrido oficial con pesos aleatorios, y justificación escrita con cifras | ganadores coinciden · tasa 0,64 ± 0,03 · 3 justificaciones ≥ 150 caracteres con ≥ 2 cifras |

> 📏 Calibración: los umbrales son los valores que mide la solución de referencia sobre el conjunto pregrabado. Las clasificaciones y recuentos son estables porque el laboratorio fija semillas y el servidor falso es determinista; las duraciones (que dependen de tu máquina) **no** se usan en ninguna rúbrica.

In [ ]:
# Instala el paquete del curso (editable) con extras de gráficos. Sirve en local y en Colab (clona el repo si hace falta).
import pathlib, subprocess, sys

def _find_root():
    for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (p / "pyproject.toml").exists() and (p / "supportops").is_dir():
            return p
    return None

ROOT = _find_root()
if ROOT is None:  # p. ej. Google Colab: el notebook se abre sin el repo
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/padillacm/IA-practice", "/content/IA-practice"], check=True)
    ROOT = pathlib.Path("/content/IA-practice/agent-observability-course")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[viz,otel]"], check=True)
if str(ROOT) not in sys.path:   # un editable recién instalado no es visible hasta reiniciar el kernel: lo añadimos a mano
    sys.path.insert(0, str(ROOT))
import importlib; importlib.invalidate_caches()
print("Curso en:", ROOT)

In [ ]:
import os
os.environ.setdefault("LLM_PROVIDER", "mock")          # mock = determinista, sin coste, sin API keys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from supportops import *                                 # SupportAgent, generate_tickets, check_stack, ...
from supportops import viz, sample_data, schema, backends
viz.apply_style()
pd.set_option("display.width", 140); pd.set_option("display.max_colwidth", 80)
SEED = 42
FAST_DEV_RUN = True   # True: pocos tickets (iteración rápida). False: tamaños «completos» del texto

import importlib.util, json, gzip, re, collections, inspect, time
from importlib import metadata
from supportops import autoinstr_lab as lab
from supportops.telemetry import setup_telemetry, OTelInstrumentor
_ = check_stack([])          # este módulo no necesita servicios de Compose

def has(mod: str) -> bool:
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def ver(pkg: str) -> str:
    try:
        return metadata.version(pkg)
    except metadata.PackageNotFoundError:
        return "no instalado"

LIBS = {"openai": has("openai"), "openlit": has("openlit"), "openllmetry": has("traceloop"),
        "official": has("opentelemetry.instrumentation.genai.openai")}
PACKAGES = ["openai", "openlit", "traceloop-sdk", "opentelemetry-instrumentation-genai-openai", "opentelemetry-sdk"]
print("Librerías para re-ejecutar el laboratorio (opcional):", LIBS)

In [ ]:
# Conjunto pregrabado: 24 tickets (semilla 42) por variante, generado con supportops.autoinstr_lab.build_dataset().
REC = ROOT / "06_autoinstrumentacion" / "recorded" / "lab_seed42.json.gz"
LAB = json.load(gzip.open(REC, "rt", encoding="utf-8"))["runs"]
print(len(LAB), "ejecuciones pregrabadas:", ", ".join(LAB)[:300], "...")
RUN_KEYS = list(LAB)

## Paso 0 · Herramientas que te damos hechas (de la lección)

Las funciones de la lección que no vas a tener que reescribir: `trace_stats`, `attr_coverage`, `ticket_of_trace`, `failure_attribution`, `token_attribution`, `pii_spans`, `kb_per_ticket`, `is_chat`, `traces_of`. Los **tickets** y su ground truth están en `LAB[<variante>]["results"]`.

In [ ]:
EXPECTED_CHAT = ["gen_ai.operation.name", "gen_ai.provider.name", "gen_ai.request.model", "gen_ai.response.model",
                 "gen_ai.response.id", "gen_ai.response.finish_reasons", "gen_ai.usage.input_tokens",
                 "gen_ai.usage.output_tokens", "gen_ai.conversation.id", "supportops.ticket.id", "supportops.step",
                 "supportops.attempt"]
EMAIL = re.compile(r"[\w.]+@example\.com")           # los PII sintéticos de SupportOps son correos @example.com

def is_chat(s: dict) -> bool:
    return s["attributes"].get("gen_ai.operation.name") == "chat"

def traces_of(spans: list[dict]) -> dict[str, list[dict]]:
    out = collections.defaultdict(list)
    for s in spans:
        out[s["trace_id"]].append(s)
    return out

def trace_stats(spans: list[dict]) -> dict:
    """Forma de las trazas: ¿cuántas hay, de qué tamaño y cuántas tienen una raíz «de agente» (no un span de LLM)?"""
    tr = traces_of(spans)
    rooted = 0
    for t in tr.values():
        roots = [s for s in t if s["parent_span_id"] is None]
        rooted += bool(roots) and not is_chat(roots[0])
    return {"n_spans": len(spans), "n_traces": len(tr), "spans_per_trace": len(spans) / max(1, len(tr)),
            "rooted_share": rooted / max(1, len(tr))}

def attr_coverage(spans: list[dict], expected: list[str] = EXPECTED_CHAT, only_ok: bool = True) -> dict[str, float]:
    """Fracción de spans `chat` (sin error) que llevan cada atributo esperado."""
    chats = [s for s in spans if is_chat(s) and (s["status"] != "ERROR" or not only_ok)]
    return {a: sum(a in s["attributes"] for s in chats) / max(1, len(chats)) for a in expected}

def ticket_of_trace(spans: list[dict]) -> dict[str, str]:
    """trace_id -> ticket, si algún span de la traza declara `supportops.ticket.id` o `gen_ai.conversation.id`."""
    m = {}
    for s in spans:
        t = s["attributes"].get("supportops.ticket.id") or s["attributes"].get("gen_ai.conversation.id")
        if t:
            m[s["trace_id"]] = t
    return m

def failure_attribution(run: dict) -> tuple[float, int]:
    """De los tickets que acabaron mal (error / max_steps / degraded), ¿qué fracción tiene una traza atribuible con algún span ERROR?"""
    bad = [r["ticket_id"] for r in run["results"] if r["status"] in ("error", "max_steps", "degraded")]
    t_of, tr = ticket_of_trace(run["spans"]), traces_of(run["spans"])
    with_err = {t_of[tid] for tid, ss in tr.items() if tid in t_of and any(s["status"] == "ERROR" for s in ss)}
    return (sum(t in with_err for t in bad) / len(bad) if bad else float("nan")), len(bad)

def token_attribution(run: dict) -> float:
    """Fracción de tickets cuyos tokens (suma de spans `chat` de su traza) coinciden con los del AgentResult."""
    t_of, tok = ticket_of_trace(run["spans"]), collections.Counter()
    for s in run["spans"]:
        if is_chat(s) and s["trace_id"] in t_of:
            a = s["attributes"]
            tok[t_of[s["trace_id"]]] += int(a.get("gen_ai.usage.input_tokens", 0)) + int(a.get("gen_ai.usage.output_tokens", 0))
    ok = sum(tok.get(r["ticket_id"], 0) == int(r["input_tokens"]) + int(r["output_tokens"]) for r in run["results"])
    return ok / len(run["results"])

def pii_spans(spans: list[dict]) -> int:
    """Nº de spans que contienen un correo (PII sintético) en atributos o eventos."""
    return sum(bool(EMAIL.search(json.dumps([s["attributes"], s["events"]], ensure_ascii=False))) for s in spans)

def kb_per_ticket(run: dict) -> float:
    return sum(len(json.dumps(s["attributes"], ensure_ascii=False)) for s in run["spans"]) / 1024 / len(run["results"])

## Paso 1 · Inventario de spans y ruido

Una forma rápida de ver «qué me ha instalado esta librería» es clasificar los spans por **categoría**.
**TODO 1.** Implementa `classify(span)` y `span_inventory(spans)`:

| Categoría | Regla |
|---|---|
| `chat` | `is_chat(span)` (operación GenAI `chat`; el nombre de span cambia según la librería: `chat mock-small`, `openai.chat`…) |
| `agent` | nombre que empieza por `invoke_agent`, o es `run`, o termina en `.agent` (decoradores de OpenLLMetry/OpenLIT) |
| `tool` | nombre que empieza por `execute_tool`, o termina en `.tool`, o es uno de `TOOL_NAMES` |
| `http` | nombre `POST` o `GET` (spans de `httpx`) |
| `other` | el resto |

`span_inventory` devuelve un `dict` con **las cinco claves** (0 si no hay).

<details>
<summary>🪜 Pista 1</summary>

Evalúa en este orden: primero `chat` (porque `openai.chat` no empieza por `chat`), luego agente, herramienta, http. `collections.Counter(classify(s) for s in spans)` y rellena con `.get(k, 0)` para las 5 claves.

</details>

<details>
<summary>🪜 Pista 2</summary>

`def classify(s): n = s['name']; if is_chat(s): return 'chat'; if n.startswith('invoke_agent') or n == 'run' or n.endswith('.agent'): return 'agent'; ...` y después `{k: c.get(k, 0) for k in ('agent','chat','tool','http','other')}`.

</details>

In [ ]:
TOOL_NAMES = {"get_order", "issue_refund", "search_kb", "escalate_to_human"}

def classify(s: dict) -> str:
    # TODO 1a
    raise NotImplementedError("TODO 1a")

def span_inventory(spans: list[dict]) -> dict:
    # TODO 1b
    raise NotImplementedError("TODO 1b")

try:
    inv = pd.DataFrame({v: span_inventory(LAB[v]["spans"]) for v in ["manual", "openlit", "openllmetry", "official", "hybrid_openllmetry", "openllmetry_decorated"]}).T
    display(inv)
except NotImplementedError as e:
    inv = None; print("Pendiente:", e)

## Paso 2 · ¿Cuántos spans `chat` por petición real? (doble instrumentación y reintentos)

Si hay una sola librería por proveedor y el SDK no reintenta por dentro, **un span `chat` = una petición real** (`meta["server_requests"]`).
**TODO 2.** `duplicate_ratio(run)` = nº de spans `chat` / peticiones reales. Interpreta: > 1 → doble instrumentación; < 1 → peticiones que no ves (reintentos ocultos).

In [ ]:
def duplicate_ratio(run: dict) -> float:
    # TODO 2
    raise NotImplementedError("TODO 2")

try:
    for k in ["official", "both", "retries2:official", "retries2:openlit"]:
        print(f"{k:20s} chat/peticiones = {duplicate_ratio(LAB[k]):.3f}")
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 3 · ¿Por qué atributo se filtra el PII?

«Hay PII en 5 spans» no basta para arreglarlo: necesitas saber **en qué atributo** (para saber qué interruptor apagar). **TODO 3.** `leak_report(run)` devuelve `{"spans": n, "emails": n_correos_distintos, "keys": {atributo: nº_de_spans}}`, mirando **todos** los atributos de cada span **y los atributos de sus eventos** (usa la clave `evento:<nombre>.<atributo>` para estos últimos). Usa `EMAIL` y `json.dumps(valor, ensure_ascii=False)` para buscar dentro de valores no-string (listas, dicts).

<details>
<summary>🪜 Pista 1</summary>

Recorre `s['attributes'].items()` y, además, `[(f"evento:{e['name']}.{k}", v) for e in s['events'] for k, v in e['attributes'].items()]`. Para cada par, `EMAIL.findall(json.dumps(v, ensure_ascii=False))`: si hay coincidencias, suma 1 a `keys[k]`, añade los correos a un `set` y marca el span como «con fuga» (una sola vez por span).

</details>

In [ ]:
def leak_report(run: dict) -> dict:
    # TODO 3
    raise NotImplementedError("TODO 3")

try:
    for k in ["openlit", "off:openlit", "openllmetry_decorated", "off:openlit_decorated", "official"]:
        print(f"{k:24s}", leak_report(LAB[k]))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 4 · Un `SpanProcessor` que enriquece y audita

En la lección el procesador copiaba el ticket activo. Ahora, además, queremos **saber qué librería creó cada span** (para detectar duplicados en producción).
**TODO 4.** Escribe `TicketContextProcessor(SpanProcessor)` con `on_start(self, span, parent_context=None)` que:
1. **siempre** añade `supportops.instrumented_by` = nombre del *instrumentation scope* del span (`span.instrumentation_scope.name`);
2. **solo si hay un ticket activo** (`lab._TICKET.get()` devuelve un `dict` con `ticket` y `tier`, o `None`), añade `supportops.ticket.id`, `gen_ai.conversation.id` (= ticket) y `supportops.customer.tier`.

El banco de pruebas `harness(processor)` simula a dos «librerías» (scopes `opentelemetry.instrumentation.genai.openai` y `opentelemetry.instrumentation.httpx`) creando spans dentro de cada ticket de la ejecución `manual` y **un span más fuera de cualquier ticket**.

<details>
<summary>🪜 Pista 1</summary>

`from opentelemetry.sdk.trace import SpanProcessor`; el hook es `on_start`, **no** `on_end` (inmutable). `lab._TICKET` es un `ContextVar`: `lab._TICKET.get()`.

</details>

<details>
<summary>🪜 Pista 2</summary>

`span.set_attribute("supportops.instrumented_by", span.instrumentation_scope.name)` y, si `t := lab._TICKET.get()`, `span.set_attributes({"supportops.ticket.id": t["ticket"], ...})`.

</details>

In [ ]:
from opentelemetry.sdk.trace import SpanProcessor, TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

class TicketContextProcessor(SpanProcessor):
    def on_start(self, span, parent_context=None):
        # TODO 4
        raise NotImplementedError("TODO 4")

def harness(proc: SpanProcessor) -> list:
    mem = InMemorySpanExporter(); tp = TracerProvider(); tp.add_span_processor(proc); tp.add_span_processor(SimpleSpanProcessor(mem))
    lib, http = tp.get_tracer("opentelemetry.instrumentation.genai.openai"), tp.get_tracer("opentelemetry.instrumentation.httpx")
    for r in LAB["manual"]["results"]:
        tok = lab._TICKET.set({"ticket": r["ticket_id"], "tier": r["tier"]})
        try:
            with lib.start_as_current_span("chat mock-small", attributes={"gen_ai.operation.name": "chat"}):
                with http.start_as_current_span("POST"):
                    pass
        finally:
            lab._TICKET.reset(tok)
    with lib.start_as_current_span("chat fuera de ticket"):
        pass
    return list(mem.get_finished_spans())

try:
    H = harness(TicketContextProcessor())
    print(len(H), "spans ·", sum("supportops.ticket.id" in s.attributes for s in H), "con ticket.id ·", sum("supportops.instrumented_by" in s.attributes for s in H), "con instrumented_by")
    print("el span fuera de ticket:", dict(H[-1].attributes))
except NotImplementedError as e:
    H = None; print("Pendiente:", e)

## Paso 5 · Una prueba de contrato para actualizar librerías sin miedo

Un *dashboard* de «tokens por modelo» depende de **nombres y valores** de atributo concretos. La prueba de contrato (en CI, con el LLM `mock`) ejecuta el agente y comprueba el **contrato** del equipo:

- **Presencia:** cada span `chat` **sin error** lleva los 8 atributos `SPEC_REQUIRED` (los primeros 8 de `EXPECTED_CHAT`).
- **Valores:** `gen_ai.response.finish_reasons` ⊆ `ALLOWED_FINISH` (`stop`, `length`, `tool_calls`: el contrato del equipo, para que las consultas funcionen con cualquier servicio).
- **Privacidad:** ningún span con PII (`pii_spans(...) == 0`).

**TODO 5.** `contract_violations(run)` devuelve un `dict` `{motivo: nº}`: `falta:<atributo>` por cada ausencia, `finish_reason_fuera_de_contrato` por cada span con algún valor fuera del conjunto, y `pii` (nº de spans con PII) si es > 0. Un dict vacío significa «cumple el contrato».

In [ ]:
SPEC_REQUIRED = EXPECTED_CHAT[:8]
ALLOWED_FINISH = {"stop", "length", "tool_calls"}

def contract_violations(run: dict, required=SPEC_REQUIRED, allowed=ALLOWED_FINISH) -> dict:
    # TODO 5
    raise NotImplementedError("TODO 5")

try:
    for k in ["official", "hybrid_official", "openlit", "openllmetry", "hybrid_openllmetry", "openllmetry_decorated"]:
        print(f"{k:24s}", contract_violations(LAB[k]))
except NotImplementedError as e:
    print("Pendiente:", e)

## Paso 6 · La decisión con tus propias medidas

Con tus funciones ya puedes puntuar siete opciones. La matriz usa **criterios medidos** (con tus funciones) y **criterios cualitativos** (dados; con su justificación en la lección):

| Criterio | Origen | Dirección |
|---|---|---|
| `contexto` | media de `token_attribution` y de `failure_attribution` | más = mejor |
| `ruido` | fracción de spans `http` + `other` (`span_inventory`) | menos = mejor |
| `pii` | spans con PII (`leak_report`) | menos = mejor |
| `contrato` | nº total de violaciones (`contract_violations`) | menos = mejor |
| `huella` | paquetes extra medidos en la lección | menos = mejor |
| `cobertura`, `esfuerzo`, `gobernanza` | cualitativos (1-5) | más = mejor |

**TODO 6a.** `weighted_scores(SC, w)` → `pd.Series` con la puntuación ponderada (pesos normalizados) ordenada de mayor a menor. **TODO 6b.** `win_rates(SC, n=20000, seed=42)` → `dict` {opción: fracción de vectores de pesos aleatorios `Dirichlet(1,…,1)` en que gana} (usa `np.random.default_rng(seed)`; igual que la lección). **TODO 6c.** Rellena `DECISION[perfil] = (opción, justificación)` con **al menos dos cifras medidas** en cada justificación.

<details>
<summary>🪜 Pista 1</summary>

6a: `w = np.asarray(w, float); w = w / w.sum()` y `(SC.values @ w)` da un vector por opción; devuélvelo como `pd.Series(..., index=SC.index).sort_values(ascending=False)`. 6b: genera `W = rng.dirichlet(np.ones(SC.shape[1]), size=n)`; la puntuación de todas las opciones para todos los vectores es `SC.values @ W.T` (opciones × n).

</details>

<details>
<summary>🪜 Pista 2</summary>

6b (continuación): el ganador de cada vector es `np.argmax(S, axis=0)`; cuenta con `np.bincount(ganadores, minlength=len(SC))/n` y construye el diccionario `{opción: fracción}`. 6c: elige la opción con `weighted_scores(SC, pesos).index[0]` y cita cifras de `SC`/`weighted_scores` en la justificación (≥ 2 números).

</details>

In [ ]:
OPTS7 = ["manual", "openlit", "openllmetry", "official", "hybrid_official", "hybrid_openllmetry", "openllmetry_decorated"]
PKGS = {"manual": 0, "openlit": 34, "openllmetry": 61, "official": 5, "hybrid_official": 5, "hybrid_openllmetry": 61, "openllmetry_decorated": 61}

def scale(values: dict, higher_better: bool = True) -> dict:
    lo, hi = min(values.values()), max(values.values())
    return {k: (5.0 if hi == lo else 1 + 4 * ((v - lo) / (hi - lo) if higher_better else (hi - v) / (hi - lo))) for k, v in values.items()}

def build_SC() -> pd.DataFrame:
    noise = lambda spans: (span_inventory(spans)["http"] + span_inventory(spans)["other"]) / max(1, len(spans))
    meas = {"contexto": scale({o: (token_attribution(LAB[o]) + failure_attribution(LAB[o])[0]) / 2 for o in OPTS7}),
            "ruido": scale({o: noise(LAB[o]["spans"]) for o in OPTS7}, False),
            "pii": scale({o: leak_report(LAB[o])["spans"] for o in OPTS7}, False),
            "contrato": scale({o: sum(contract_violations(LAB[o]).values()) for o in OPTS7}, False),
            "huella": scale(PKGS, False)}
    qual = {"cobertura": [1, 5, 4, 3, 3, 4, 4], "esfuerzo": [1, 5, 5, 4, 3, 3, 4], "gobernanza": [5, 3, 3, 4, 4, 3, 3]}
    return pd.DataFrame({**meas, **{k: dict(zip(OPTS7, v)) for k, v in qual.items()}}).loc[OPTS7]

PROFILES = {   # pesos en el orden de las columnas de SC
    "SupportOps (agente propio, 1 proveedor, datos sensibles)": [5, 2, 5, 4, 2, 2, 3, 3],
    "Plataforma interna (muchos frameworks y equipos)":          [3, 2, 3, 4, 2, 5, 4, 3],
    "Prototipo de 1 día":                                         [1, 1, 1, 1, 2, 3, 5, 1],
}

def weighted_scores(SC: pd.DataFrame, w) -> pd.Series:
    # TODO 6a
    raise NotImplementedError("TODO 6a")

def win_rates(SC: pd.DataFrame, n: int = 20000, seed: int = 42) -> dict:
    # TODO 6b
    raise NotImplementedError("TODO 6b")

DECISION: dict[str, tuple[str, str]] = {}      # TODO 6c: perfil -> (opción, justificación con cifras)

try:
    SC = build_SC(); display(SC.round(2))
    for name, w in PROFILES.items():
        print(name, "->", weighted_scores(SC, w).round(2).head(3).to_dict())
    WR = win_rates(SC); print("tasa de victorias con pesos aleatorios:", {k: round(v, 3) for k, v in WR.items()})
except NotImplementedError as e:
    SC = WR = None; print("Pendiente:", e)

## Autocomprobación (rúbrica)
Ejecuta esta celda para ver qué entregables pasan. Los valores esperados son los de la solución de referencia.

In [ ]:
def check(name, fn):
    try:
        ok, detail = fn()
    except NotImplementedError:
        ok, detail = False, "pendiente"
    except Exception as e:                                  # un error tuyo no debe romper la rúbrica
        ok, detail = False, f"{type(e).__name__}: {e}"
    print(("✅" if ok else "❌"), name, "·", detail); return ok

def r1():
    exp = {"openlit": {"agent": 0, "chat": 66, "tool": 0, "http": 66, "other": 0}, "hybrid_openllmetry": {"agent": 24, "chat": 66, "tool": 36, "http": 0, "other": 0},
           "openllmetry_decorated": {"agent": 24, "chat": 66, "tool": 28, "http": 0, "other": 0}}
    n = sum(span_inventory(LAB[k]["spans"]) == v for k, v in exp.items()); return n == 3, f"{n}/3 inventarios exactos"
def r2():
    exp = {"both": 2.0, "official": 1.0, "retries2:official": 0.686}
    n = sum(abs(duplicate_ratio(LAB[k]) - v) <= 0.005 for k, v in exp.items()); return n == 3, f"{n}/3 ratios dentro de ±0,005"
def r3():
    exp = {"openllmetry_decorated": {"spans": 8, "emails": 3, "keys": {"traceloop.entity.input": 3, "gen_ai.input.messages": 5}},
           "off:openlit_decorated": {"spans": 3, "emails": 3, "keys": {"function.args": 3}}, "official": {"spans": 0, "emails": 0, "keys": {}}}
    n = sum(leak_report(LAB[k]) == v for k, v in exp.items()); return n == 3, f"{n}/3 informes exactos"
def r4():
    H = harness(TicketContextProcessor()); t = sum("supportops.ticket.id" in s.attributes for s in H); b = sum("supportops.instrumented_by" in s.attributes for s in H)
    return (len(H), t, b) == (49, 48, 49) and "supportops.ticket.id" not in H[-1].attributes, f"{t}/{len(H)} con ticket.id, {b}/{len(H)} con instrumented_by"
def r5():
    exp = {"openlit": {"pii": 5}, "openllmetry": {"finish_reason_fuera_de_contrato": 26, "pii": 5}, "official": {}, "hybrid_official": {}}
    n = sum(contract_violations(LAB[k]) == v for k, v in exp.items()); return n == 4, f"{n}/4 contratos exactos"
def r6():
    SC_ = build_SC(); tops = {p: weighted_scores(SC_, w).index[0] for p, w in PROFILES.items()}
    exp_tops = ["hybrid_official", "hybrid_official", "official"]
    wr = win_rates(SC_)
    just = [v[1] for v in DECISION.values() if isinstance(v, tuple) and len(v) == 2]
    ok_text = len(just) >= 3 and all(len(t) >= 150 and len(re.findall(r"\d+[.,]?\d*", t)) >= 2 for t in just)
    ok = list(tops.values()) == exp_tops and abs(wr["hybrid_official"] - 0.64) <= 0.03 and abs(sum(wr.values()) - 1) < 1e-9 and ok_text
    return ok, f"ganadores {list(tops.values())} · híbrido gana {wr['hybrid_official']:.2f} · justificaciones con cifras: {ok_text}"

results = [check("R1 span_inventory", r1), check("R2 duplicate_ratio", r2), check("R3 leak_report", r3), check("R4 TicketContextProcessor", r4),
           check("R5 contract_violations", r5), check("R6 decisión", r6)]
print(f"\n{sum(results)}/6 entregables correctos")

---
# ⛔ SPOILER — intenta resolverlo primero

Abajo está la solución de referencia completa (con la que se calibró la rúbrica). Si te atascas en un TODO, abre primero las pistas 🪜 del paso correspondiente.

## Solución de referencia

In [ ]:
# ---- Pasos 1 a 3
def classify(s: dict) -> str:
    n = s["name"]
    if is_chat(s):
        return "chat"
    if n.startswith("invoke_agent") or n == "run" or n.endswith(".agent"):
        return "agent"
    if n.startswith("execute_tool") or n.endswith(".tool") or n in TOOL_NAMES:
        return "tool"
    if n in ("POST", "GET"):
        return "http"
    return "other"

def span_inventory(spans: list[dict]) -> dict:
    c = collections.Counter(classify(s) for s in spans)
    return {k: c.get(k, 0) for k in ("agent", "chat", "tool", "http", "other")}

def duplicate_ratio(run: dict) -> float:
    return sum(is_chat(s) for s in run["spans"]) / run["meta"]["server_requests"]

def leak_report(run: dict) -> dict:
    keys, emails, n = collections.Counter(), set(), 0
    for s in run["spans"]:
        items = list(s["attributes"].items()) + [(f"evento:{e['name']}.{k}", v) for e in s["events"] for k, v in e["attributes"].items()]
        hit = False
        for k, v in items:
            found = EMAIL.findall(json.dumps(v, ensure_ascii=False))
            if found:
                keys[k] += 1; emails.update(found); hit = True
        n += hit
    return {"spans": n, "emails": len(emails), "keys": dict(keys)}

In [ ]:
# ---- Paso 4: procesador
class TicketContextProcessor(SpanProcessor):
    def on_start(self, span, parent_context=None):
        span.set_attribute("supportops.instrumented_by", span.instrumentation_scope.name)      # siempre: ¿qué librería lo creó?
        t = lab._TICKET.get()
        if t:                                                                                  # solo dentro de un ticket
            span.set_attributes({"supportops.ticket.id": t["ticket"], "gen_ai.conversation.id": t["ticket"], "supportops.customer.tier": t["tier"]})

H = harness(TicketContextProcessor())
print(len(H), "spans ·", sum("supportops.ticket.id" in s.attributes for s in H), "con ticket.id ·", sum("supportops.instrumented_by" in s.attributes for s in H), "con instrumented_by")

In [ ]:
# ---- Paso 5: contrato
def contract_violations(run: dict, required=SPEC_REQUIRED, allowed=ALLOWED_FINISH) -> dict:
    out = collections.Counter()
    for s in run["spans"]:
        if is_chat(s) and s["status"] != "ERROR":
            for a in required:
                if a not in s["attributes"]:
                    out[f"falta:{a}"] += 1
            fr = s["attributes"].get("gen_ai.response.finish_reasons")
            if fr is not None and any(x not in allowed for x in fr):
                out["finish_reason_fuera_de_contrato"] += 1
    n = pii_spans(run["spans"])
    if n:
        out["pii"] = n
    return dict(out)

# ---- Paso 6: matriz y sensibilidad
def weighted_scores(SC: pd.DataFrame, w) -> pd.Series:
    w = np.asarray(w, float); w = w / w.sum()
    return pd.Series(SC.values @ w, index=SC.index).sort_values(ascending=False)

def win_rates(SC: pd.DataFrame, n: int = 20000, seed: int = 42) -> dict:
    W = np.random.default_rng(seed).dirichlet(np.ones(SC.shape[1]), n)
    c = collections.Counter(SC.index[(SC.values @ W.T).argmax(axis=0)])
    return {o: c.get(o, 0) / n for o in SC.index}

SC = build_SC(); WR = win_rates(SC)
DECISION = {
    "SupportOps (agente propio, 1 proveedor, datos sensibles)": ("hybrid_official",
        "Híbrido oficial: 100 % de los tickets con tokens y fallos atribuibles (como el manual) pero con el span chat mantenido por OpenTelemetry; 0 spans con PII y 0 violaciones de contrato por defecto, solo 5 paquetes extra y gana en el 64 % de los pesos aleatorios."),
    "Plataforma interna (muchos frameworks y equipos)": ("hybrid_official",
        "También híbrido oficial (4,17 frente a 3,86 de la oficial pura): la amplitud de OpenLIT/OpenLLMetry (66 trazas huérfanas para 24 tickets y PII en 5 spans por defecto) no compensa; si hace falta más cobertura se añade OpenLLMetry con contenido apagado y el mismo SpanProcessor."),
    "Prototipo de 1 día": ("official",
        "La oficial pura (3,89) roza a OpenLIT (3,83): 1 línea, 0 PII y 0 violaciones del contrato de atributos; sirve para un prototipo, aceptando que habrá 66 trazas huérfanas y sin atribución por ticket."),
}
for p, (o, j) in DECISION.items():
    print(f"{p}\n   -> {o}: {j}\n")

In [ ]:
# Verificación final de la solución con la misma rúbrica
results = [check("R1 span_inventory", r1), check("R2 duplicate_ratio", r2), check("R3 leak_report", r3), check("R4 TicketContextProcessor", r4),
           check("R5 contract_violations", r5), check("R6 decisión", r6)]
print(f"\n{sum(results)}/6 entregables correctos")

## Retos extra (nivel experto)

1. **Reproduce en vivo.** Con las librerías instaladas: `d = lab.run_variant("openlit", n=24, content="off", enrich=True)` y pasa tus funciones sobre `d`. ¿Cuántas violaciones de contrato quedan? ¿Qué spans `http` siguen ahí? (Pista: mira `lab.run_variant` y `disabled_instrumentors` en la documentación de OpenLIT.)
2. **Guardia de doble instrumentación.** Escribe una función de arranque que cuente cuántas capas de *wrapper* tiene `openai.resources.chat.completions.Completions.create` (los parches de `wrapt` exponen `__wrapped__`) y falle si hay más de una librería. Pruébala con `lab.run_variant(..., extra="official")` en un subproceso.
3. **Prueba de contrato en `pytest`.** Convierte `contract_violations` en un test parametrizado por variante que se ejecute en CI con `mock` y falle si `pii` > 0 o si cambia el conjunto de atributos de `chat`.
4. **Pesos honestos.** Pide a dos compañeros que puntúen (sin ver tu matriz) los tres criterios cualitativos y recalcula `win_rates`. ¿Cambia el ganador? ¿Cuánto varía la tasa del híbrido?
5. **Streaming.** Extiende `lab.run_variant` para ejecutar un agente con `stream=True` e `include_usage=True` y comprueba con tu `token_attribution` que se recuperan los tokens.

## Reflexión (conexión con producción y con el hilo conductor)

- Tu prueba de contrato bloquea una actualización de OpenLIT porque cambia un valor de `finish_reasons`. ¿Quién decide si se acepta el cambio y cómo se migran los *dashboards* y las alertas existentes sin romper el histórico? (módulos 10 y 16)
- ¿Qué parte de lo medido aquí se convierte en **política del Collector** (redacción, eliminación de atributos, normalización) y no en código de cada servicio? (módulos 05 y 20)
- Un equipo vecino usa OpenLIT con captura de contenido porque sus *evals* lo necesitan (módulo 14). ¿Cómo separas, a nivel de traza, el contenido sensible del resto de la telemetría para que pueda retenerse menos tiempo?
- ¿Qué consultas de TraceQL (módulo 07) te habría gustado poder escribir y que **no** puedes escribir con la telemetría de solo-librería? Anótalas: son tu lista de atributos obligatorios.
- Conexión con SupportOps: el híbrido que has elegido es la base de instrumentación que usarán los módulos 07 (Tempo), 08 (Loki) y 13 (costes por ticket).